# 03 - Figure 3: order and transport

**Question.** How is collective order associated with run persistence,
displacement, and centroid speed?

| Panel | Analysis |
|---|---|
| A | Duration survivor by early-order tercile |
| B | Length survivor by early-order tercile |
| C | Joint order-speed phenotype coloured by run duration |

The state classification and bootstrap unit are made explicit below.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## 1. Load the centroid-run population

In [ ]:
from analysis.levy_paper.scripts import create_final_figure3_order_transport as figure3

run_cache = cache_path("centroid_order_runs")
resolved_mode = resolve_data_mode(DATA_MODE, [run_cache])
cache_ready = resolved_mode == "cache"
print("resolved_data_mode", resolved_mode)
display(show_file_status([run_cache]))

raw_runs = None
runs = None
if cache_ready:
    raw_runs = pd.read_parquet(run_cache)
    runs = figure3.load_runs()
    display(pd.DataFrame([
        {"stage": "cached rows", "rows": len(raw_runs)},
        {"stage": "centroid, complete-case and physical-range filters", "rows": len(runs)},
    ]))

## 2. Define early-order states

In [ ]:
if runs is not None:
    q1, q2 = runs["p_early_3s"].quantile([1 / 3, 2 / 3]).to_numpy(float)
    state_bounds = pd.DataFrame([
        {"state": "Low", "lower": runs["p_early_3s"].min(), "upper": q1},
        {"state": "Mid", "lower": q1, "upper": q2},
        {"state": "High", "lower": q2, "upper": runs["p_early_3s"].max()},
    ])
    display(state_bounds)
    display(order_state_summary(runs))

## 3. Cluster bootstrap and survivor grids

In [ ]:
duration_grid = length_grid = None
if runs is not None:
    cluster_cols = figure3.best_cluster_cols(runs)
    duration_grid = np.arange(1.0, min(75.0, max(35.0, runs["duration_s"].quantile(0.995))) + 1.0, 2.0)
    length_grid = np.linspace(1.0, min(85.0, max(25.0, runs["run_length_m"].quantile(0.995))), 34)
    print("bootstrap_cluster_columns", cluster_cols)
    print("bootstrap_cluster_count", runs[cluster_cols].drop_duplicates().shape[0])
    display(pd.DataFrame({"duration_grid_s": pd.Series(duration_grid), "length_grid_m": pd.Series(length_grid)}).head(12))

## 4. Inspect empirical survivor values by state

In [ ]:
if runs is not None:
    survivor_rows = []
    for state in figure3.STATE_ORDER:
        subset = runs.loc[runs["early_order_state"].astype(str).eq(state)]
        for age, estimate in zip(duration_grid[:10], figure3.ccdf(subset["duration_s"], duration_grid[:10])):
            survivor_rows.append({"state": state, "duration_s": age, "survival": estimate, "runs": len(subset)})
    display(pd.DataFrame(survivor_rows).head(15))

## 5. Inspect the order-speed phenotype inputs

In [ ]:
if runs is not None:
    phenotype_summary = runs.groupby("early_order_state", observed=True).agg(
        runs=("duration_s", "size"),
        mean_polarisation=("p_mean", "mean"),
        mean_centroid_speed_mps=("v_mean_mps", "mean"),
        median_duration_s=("duration_s", "median"),
    )
    display(phenotype_summary)

## 6. Construct the publication figure

In [ ]:
fig = None
created = []
if BUILD_FIGURE and runs is not None:
    figure3.configure_paper_plotting(base=figure3.PAPER_FONT_BASE)
    fig, created = figure3.build_publication_figure(
        runs, duration_grid, length_grid, cluster_cols,
        close_figure=False,
        save_outputs=SAVE_FIGURE_OUTPUTS,
    )
elif BUILD_FIGURE:
    print("Processed order-run cache unavailable; using frozen Figure 3.")

display_mode = show_figure(fig, FINAL_FIGURES / "figure3_order_transport.png")
print("figure_display_mode", display_mode)